In [12]:
import os
from kaggle_secrets import UserSecretsClient


REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"

token = UserSecretsClient().get_secret(
    "GITHUB_TOKEN"
)

repo_url = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)


# Clone once; otherwise force Kaggle copy
# to current GitHub main.
if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    !git clone {repo_url} {REPO_PATH}

else:
    !git -C {REPO_PATH} fetch origin main
    !git -C {REPO_PATH} reset --hard origin/main


os.chdir(
    REPO_PATH
)


import sys

SRC_PATH = os.path.join(
    REPO_PATH,
    "src",
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH,
    )

!git log -1 --oneline

print(
    "Code synced and package installed."
)

From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD
HEAD is now at 1315ccb Add diffusion portfolio test evaluation
1315ccb (HEAD -> main, origin/main, origin/HEAD) Add diffusion portfolio test evaluation
Code synced and package installed.


In [9]:
!pip install -e .

Obtaining file:///kaggle/working/diffusion-portfolio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusion-portfolio (pyproject.toml) ... done
  Created wheel for diffusion-portfolio: filename=diffusion_portfolio-0.1.0-0.editable-py3-none-any.whl size=1548 sha256=7b2583c0d895f00f320f02969d476cacb13c362a94eca440225617ebc4d642e9
  Stored in directory: /tmp/pip-ephem-wheel-cache-6ck4m9hq/wheels/84/80/56/dface83323a0e9e6034b63f9965eba73644490018b8040335d
Successfully built diffusion-portfolio
  Attempting uninstall: diffusion-portfolio
    Found existing installation: diffusion-portfolio 0.1.0
    Uninstalling diffusion-portfolio-0.1.0:
      Successfully uninstalled diffusion-portfolio-0.1.0


In [14]:
!python scripts/train_diffusion.py \
    --config configs/mvp.yaml \
    --output-dir /kaggle/working/vanilla_vpred \
    --device cuda

VANILLA CONDITIONAL DIFFUSION TRAINING
Device: cuda
GPU: Tesla T4
Prediction type: v_prediction
Diffusion steps: 100

Loading KF12 data...
Train samples: 10516
Validation samples: 1256
Test samples: 4781
Parameters: 1,044,417

Training...
Epoch 001/050 | train=0.843010 | val=1.615484
Epoch 002/050 | train=0.740929 | val=1.641421
Epoch 003/050 | train=0.716428 | val=1.579357
Epoch 004/050 | train=0.720818 | val=1.621627
Epoch 005/050 | train=0.698276 | val=1.600232
Epoch 006/050 | train=0.693718 | val=1.618063
Epoch 007/050 | train=0.701392 | val=1.608534
Epoch 008/050 | train=0.679560 | val=1.598500
Epoch 009/050 | train=0.694375 | val=1.582620
Epoch 010/050 | train=0.695771 | val=1.605567
Epoch 011/050 | train=0.661530 | val=1.624049
Epoch 012/050 | train=0.645514 | val=1.616791
Epoch 013/050 | train=0.669309 | val=1.610224
Epoch 014/050 | train=0.643770 | val=1.607395
Epoch 015/050 | train=0.655873 | val=1.665431
Epoch 016/050 | train=0.652135 | val=1.645180
Epoch 017/050 | train=0.6

In [5]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"

SRC_PATH = os.path.join(
    REPO_PATH,
    "src",
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH,
    )

print(SRC_PATH)

import diffusion_portfolio

print("Import works.")

/kaggle/working/diffusion-portfolio/src
Import works.


In [6]:
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from diffusion_portfolio.config import load_config
from diffusion_portfolio.data import (
    TrainStandardizer,
    build_window_datasets,
    collate_return_batch,
    load_daily_risk_free,
    load_kf12_daily,
    slice_return_table,
    to_excess_returns,
)
from diffusion_portfolio.models.diffusion import (
    ConditionalDiffusionModel,
)


OUTPUT_DIR = Path(
    "/kaggle/working/vanilla_vpred"
)

cfg = load_config(
    OUTPUT_DIR / "config.yaml"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# Rebuild the exact data pipeline
# ============================================================

assets = slice_return_table(
    load_kf12_daily(),
    start=cfg.data.sample_start,
    end=cfg.data.sample_end,
)

risk_free = load_daily_risk_free()

excess = to_excess_returns(
    assets,
    risk_free,
)

scaler = TrainStandardizer.fit(
    excess,
    train_end=cfg.data.train_end,
)

scaled = scaler.transform(
    excess
)

datasets = build_window_datasets(
    scaled,
    excess,
    lookback=cfg.data.lookback,
    horizon=cfg.data.horizon,
    train_end=cfg.data.train_end,
    val_end=cfg.data.val_end,
    test_end=cfg.data.sample_end,
)


# Evaluation loaders must NOT shuffle.
train_eval_loader = DataLoader(
    datasets.train,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_return_batch,
)

val_eval_loader = DataLoader(
    datasets.val,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_return_batch,
)


# ============================================================
# Restore the exact best-validation checkpoint
# ============================================================

model = ConditionalDiffusionModel(
    lookback=cfg.data.lookback,
    n_assets=12,
    condition_dim=cfg.model.condition_dim,
    history_hidden_dim=(
        cfg.model.history_hidden_dim
    ),
    diffusion_steps=(
        cfg.model.diffusion_steps
    ),
    schedule_type=cfg.model.schedule,
    prediction_type=(
        cfg.model.prediction_type
    ),
    channels=list(
        cfg.model.channels
    ),
    time_embed_dim=(
        cfg.model.time_embed_dim
    ),
    n_res_blocks=(
        cfg.model.n_res_blocks
    ),
).to(device)

checkpoint = torch.load(
    OUTPUT_DIR / "best.pt",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Checkpoint epoch:",
    checkpoint["epoch"],
)

print(
    "Checkpoint saved val loss:",
    checkpoint["val_loss"],
)


# ============================================================
# Compare trained model vs zero-v predictor
# using IDENTICAL timesteps + noise.
# ============================================================

@torch.no_grad()
def diagnose_split(
    name,
    loader,
):
    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        cfg.training.validation_seed
    )

    model_sse = 0.0
    zero_sse = 0.0
    n_elements = 0

    for batch in loader:
        history = batch[
            "history"
        ].to(device)

        target = batch[
            "target"
        ].to(device)

        x0 = target[:, 0, :]

        batch_size = x0.shape[0]

        timesteps = torch.randint(
            low=0,
            high=model.diffusion_steps,
            size=(batch_size,),
            generator=generator,
            device="cpu",
            dtype=torch.long,
        ).to(device)

        noise = torch.randn(
            x0.shape,
            generator=generator,
            device="cpu",
            dtype=x0.dtype,
        ).to(device)

        output = model.training_loss(
            history,
            target,
            timesteps=timesteps,
            noise=noise,
        )

        training_target = (
            output.training_target
        )

        prediction = (
            output.prediction
        )

        model_sse += F.mse_loss(
            prediction,
            training_target,
            reduction="sum",
        ).item()

        # Trivial predictor:
        #       v_hat = 0
        zero_sse += (
            training_target
            .square()
            .sum()
            .item()
        )

        n_elements += (
            training_target.numel()
        )

    model_loss = (
        model_sse
        / n_elements
    )

    zero_loss = (
        zero_sse
        / n_elements
    )

    skill = (
        1.0
        - model_loss
        / zero_loss
    )

    print()
    print("=" * 64)
    print(name)
    print("=" * 64)

    print(
        "Model loss:",
        model_loss,
    )

    print(
        "Zero-v baseline loss:",
        zero_loss,
    )

    print(
        "Skill vs zero:",
        skill,
    )

    print(
        "Improvement (%):",
        100.0 * skill,
    )

    return {
        "model_loss": model_loss,
        "zero_loss": zero_loss,
        "skill": skill,
    }


train_result = diagnose_split(
    "TRAIN",
    train_eval_loader,
)

val_result = diagnose_split(
    "VALIDATION",
    val_eval_loader,
)


print()
print("=" * 64)
print("SUMMARY")
print("=" * 64)

print(
    "Zero-loss val/train ratio:",
    (
        val_result["zero_loss"]
        / train_result["zero_loss"]
    ),
)

print(
    "Model-loss val/train ratio:",
    (
        val_result["model_loss"]
        / train_result["model_loss"]
    ),
)

Device: cuda
Checkpoint epoch: 3
Checkpoint saved val loss: 1.579357321474962

TRAIN
Model loss: 0.7082894594780889
Zero-v baseline loss: 0.9892535018001928
Skill vs zero: 0.2840162221420697
Improvement (%): 28.401622214206967

VALIDATION
Model loss: 1.5793572768031159
Zero-v baseline loss: 1.847008642370787
Skill vs zero: 0.14491072723088005
Improvement (%): 14.491072723088006

SUMMARY
Zero-loss val/train ratio: 1.8670731405142316
Model-loss val/train ratio: 2.2298189753760886


In [8]:
!python scripts/evaluate_diffusion_forecasts.py \
    --run-dir /kaggle/working/vanilla_vpred \
    --batch-size 32 \
    --device cuda

DIFFUSION TEST FORECAST EVALUATION
Device: cuda
GPU: Tesla T4
Scenarios per date: 256
Inference batch size: 32

Test samples: 4781
Test start: 2005-01-03 00:00:00
Test end: 2023-12-29 00:00:00

Checkpoint epoch: 3
Checkpoint validation loss: 1.579357321474962

Generating test scenarios...
Sampling: 100%|███████████████████████████████| 150/150 [09:21<00:00,  3.74s/it]

Saved scenarios: /kaggle/working/vanilla_vpred/test_scenarios.npz
Scenario cube: (4781, 256, 12)
Scenario cube size (MiB): 56.02734375

Raw scenario mean: -0.0009685299592092633
Raw scenario std: 0.008783436380326748
Raw scenario max abs: 0.06748927384614944

Computing probabilistic metrics...

TEST PROBABILISTIC EVALUATION
CRPS mean: 0.007408589419520529
CRPS std across assets: 0.0016754970381537568
Energy Score: 0.029721776550272312

Calibration:
   level         PICP          ACE
    0.50     0.387977    -0.112023
    0.80     0.650387    -0.149613
    0.90     0.762009    -0.137991
    0.95     0.831155    -0.118845


In [13]:
!python scripts/evaluate_diffusion_portfolios.py \
    --run-dir /kaggle/working/vanilla_vpred

DIFFUSION PORTFOLIO EVALUATION
Scenario cube: (4781, 256, 12)
Dates: 2005-01-03 00:00:00 to 2023-12-29 00:00:00

diffusion_min_variance
diffusion_min_variance: 100%|███████████████| 4781/4781 [03:32<00:00, 22.54it/s]
Frictionless Sharpe: 0.4399038355506292
5-day gross Sharpe: 0.45340273681897203
5-day net Sharpe: 0.4164017509171397
5-day monthly turnover: 0.5681946863053878
5-day total transaction cost: 0.12954838847762845

diffusion_tangency
diffusion_tangency: 100%|██████████████████| 4781/4781 [00:41<00:00, 115.60it/s]
Frictionless Sharpe: 0.3952226370391463
5-day gross Sharpe: 0.49858347223634925
5-day net Sharpe: 0.3442777750444977
5-day monthly turnover: 3.270238276761375
5-day total transaction cost: 0.7456143271015936

diffusion_mean_cvar
diffusion_mean_cvar: 100%|█████████████████| 4781/4781 [00:37<00:00, 128.85it/s]
Frictionless Sharpe: 0.4704708007306257
5-day gross Sharpe: 0.45925343529218987
5-day net Sharpe: 0.35920910240816695
5-day monthly turnover: 1.562144575944716
5-

In [15]:
from pathlib import Path

RUN_DIR = Path("/kaggle/working/vanilla_vpred")

for path in sorted(RUN_DIR.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name:40s} {size_mb:8.2f} MiB")

best.pt                                     12.05 MiB
config.yaml                                  0.00 MiB
history.csv                                  0.00 MiB
standardizer.npz                             0.00 MiB
summary.json                                 0.00 MiB
test_diffusion_weights.npz                   0.69 MiB
test_portfolio_metrics.json                  0.00 MiB
test_probabilistic_metrics.json              0.00 MiB
test_scenarios.npz                          56.28 MiB


In [16]:
import shutil

shutil.make_archive(
    "/kaggle/working/vanilla_vpred_stage5",
    "zip",
    "/kaggle/working/vanilla_vpred",
)

!ls -lh /kaggle/working/vanilla_vpred_stage5.zip

-rw-r--r-- 1 root root 64M Sep 29 10:03 /kaggle/working/vanilla_vpred_stage5.zip
